# Module X3: exercises

These use `app_trial.csv` (conversions per group), `students.csv` (exam scores) and `service_desk.csv` (tickets). Random draws are seeded and the checks allow for Monte Carlo error.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Beta–Binomial update.** Write `beta_posterior(x, n, a0=1, b0=1)` returning the posterior parameters `(a, b)` $=(a_0+x,\ b_0+n-x)$ and `credible_interval(a, b, level=0.95)` returning the **equal-tailed** interval from `stats.beta.ppf`.
For the control group (17 conversions of 100, flat prior) store `a_c, b_c`, the posterior mean `mean_c` and the interval `ci_c`.

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import stats

trial = pd.read_csv(DATA / "app_trial.csv")
students = pd.read_csv(DATA / "students.csv")
desk = pd.read_csv(DATA / "service_desk.csv")

def beta_posterior(x, n, a0=1, b0=1):
    # YOUR CODE HERE
    pass

def credible_interval(a, b, level=0.95):
    # YOUR CODE HERE
    pass

a_c = b_c = mean_c = ci_c = None   # YOUR CODE HERE

In [ ]:
assert beta_posterior(3, 10) == (4, 8) and beta_posterior(3, 10, 2, 2) == (5, 9)
assert (a_c, b_c) == (18, 84) and abs(mean_c - 0.17647) < 1e-4
assert abs(ci_c[0] - 0.1092) < 1e-3 and abs(ci_c[1] - 0.2558) < 1e-3, ci_c
print(f"✅ Correct! Posterior Beta({a_c}, {b_c}): mean {mean_c:.3f}, 95% credible interval ({ci_c[0]:.3f}, {ci_c[1]:.3f})")

## Exercise 2

**Prior sensitivity.** With the priors flat `(1, 1)`, weak `(2, 8)` and strong `(20, 80)`, compute the posterior **means** for a small sample (2 successes of 10) as the list `small_means` and for a larger one (17 of 100) as `large_means` (in that prior order).
Then store the spread (max − min) of each list as `spread_small`, `spread_large`.

In [ ]:
priors = [(1, 1), (2, 8), (20, 80)]
small_means = large_means = None         # YOUR CODE HERE
spread_small = spread_large = None       # YOUR CODE HERE

In [ ]:
assert np.allclose(small_means, [0.25, 0.20, 0.20]) and np.allclose(large_means, [0.17647, 0.17273, 0.185], atol=1e-4)
assert spread_small > 3 * spread_large, "the prior matters much more with 10 observations than with 100"
print(f"✅ Correct! spread of posterior means: {spread_small:.3f} with n = 10 vs {spread_large:.3f} with n = 100")

## Exercise 3

**Grid approximation.** Write `grid_posterior(x, n, prior_density, n_grid=2000)` returning `(grid, post)` where `grid` is `np.linspace(0.0005, 0.9995, n_grid)` and `post` is `prior_density(grid) * stats.binom.pmf(x, n, grid)` **normalised to sum to 1**.
Compute `mean_flat` (flat prior) and `mean_informed` for the prior `lambda p: np.exp(-0.5 * ((p - 0.30) / 0.05) ** 2)`, both for 17 of 100.

In [ ]:
def grid_posterior(x, n, prior_density, n_grid=2000):
    # YOUR CODE HERE
    pass

mean_flat = mean_informed = None   # YOUR CODE HERE

In [ ]:
g, p = grid_posterior(17, 100, lambda q: np.ones_like(q))
assert abs(p.sum() - 1) < 1e-12 and len(g) == 2000
assert abs(mean_flat - 18 / 102) < 1e-4, "a flat prior reproduces the conjugate Beta(18, 84) mean"
assert 0.17 < mean_informed < 0.30 and abs(mean_informed - 0.225) < 0.003, mean_informed
print(f"✅ Correct! flat prior mean {mean_flat:.4f}; with a prior near 0.30 the posterior mean moves to {mean_informed:.3f}, between the data (0.17) and the prior (0.30)")

## Exercise 4

**Bayesian A/B test.** Write `prob_a_beats_b(xa, na, xb, nb, n_draws=200_000, seed=0)`: draw from each group's flat-prior Beta posterior and return the share of draws where A's rate exceeds B's.
Write `prob_diff_above(xa, na, xb, nb, delta, n_draws=200_000, seed=0)` for $P(p_A-p_B>\delta)$. Apply them to gamified (33/100) vs control (17/100) as `p_better` and `p_gt_5pts` (δ = 0.05).

In [ ]:
def prob_a_beats_b(xa, na, xb, nb, n_draws=200_000, seed=0):
    # YOUR CODE HERE
    pass

def prob_diff_above(xa, na, xb, nb, delta, n_draws=200_000, seed=0):
    # YOUR CODE HERE
    pass

p_better = p_gt_5pts = None   # YOUR CODE HERE

In [ ]:
assert abs(prob_a_beats_b(20, 100, 20, 100) - 0.5) < 0.01, "identical data give a coin-flip"
assert abs(p_better - 0.9954) < 0.003, p_better
assert abs(p_gt_5pts - 0.9625) < 0.005, p_gt_5pts
print(f"✅ Correct! P(gamified better) = {p_better:.3f}; P(better by more than 5 points) = {p_gt_5pts:.3f}")

## Exercise 5

**Normal–Normal update.** Write `normal_posterior(mu0, tau, xbar, sigma, n)` returning `(post_mean, post_sd)` where precision $=1/\tau^2+n/\sigma^2$ and $\text{post\_mean}=(\mu_0/\tau^2+n\bar x/\sigma^2)/\text{precision}$.
Apply it to the exam scores with prior $N(65,3^2)$, taking $\sigma$ as the sample standard deviation, as `pm, ps`.

In [ ]:
xs = students["exam_score"].to_numpy()
def normal_posterior(mu0, tau, xbar, sigma, n):
    # YOUR CODE HERE
    pass

pm = ps = None   # YOUR CODE HERE

In [ ]:
m, sd = normal_posterior(0, 1e6, 10.0, 2.0, 25)                      # an extremely vague prior
assert abs(m - 10.0) < 1e-6 and abs(sd - 2.0 / 5) < 1e-6, "a vague prior returns the sample mean and the standard error"
assert abs(pm - 67.767) < 0.005 and abs(ps - 0.721) < 0.002, (pm, ps)
assert xs.mean() > pm > 65, "the posterior mean lies between the prior mean and the sample mean"
print(f"✅ Correct! Posterior N({pm:.2f}, {ps:.3f}): the data (mean {xs.mean():.2f}) dominate the prior (65)")

## Exercise 6

**Gamma–Poisson update.** Write `gamma_poisson_posterior(a0, b0, counts)` returning `(a, b)` $=(a_0+\sum y,\ b_0+n)$ (shape, **rate**). With the prior Gamma(2, 0.5), the tickets per hour give `a_t, b_t`;
store the posterior mean `mean_t` ($a/b$) and the 95% credible interval `ci_t` (SciPy's gamma uses `scale = 1 / rate`).

In [ ]:
def gamma_poisson_posterior(a0, b0, counts):
    # YOUR CODE HERE
    pass

tickets = desk["tickets"].to_numpy()
a_t = b_t = mean_t = ci_t = None   # YOUR CODE HERE

In [ ]:
assert gamma_poisson_posterior(1, 1, [2, 3, 4]) == (10, 4)
assert (a_t, b_t) == (2100, 480.5) and abs(mean_t - 4.3704) < 1e-3
assert abs(ci_t[0] - 4.1855) < 2e-3 and abs(ci_t[1] - 4.5593) < 2e-3, ci_t
print(f"✅ Correct! Rate {mean_t:.3f} tickets/hour, 95% credible interval ({ci_t[0]:.3f}, {ci_t[1]:.3f})")

## Exercise 7

**A Metropolis sampler.** Write `metropolis_1d(log_target, start, step, n_iter, seed)` returning `(samples, acceptance_rate)`: at each iteration propose `current + N(0, step²)`, accept if `log(u) < log_target(proposal) - log_target(current)` with `u ~ Uniform(0,1)`,
otherwise keep the current value (the kept value is recorded either way). Test it on a standard normal target: `log_target = lambda z: -0.5 * z**2`, start 0, step 2.4, 20,000 iterations, as `draws, rate`.

In [ ]:
def metropolis_1d(log_target, start, step, n_iter, seed):
    # YOUR CODE HERE
    pass

draws, rate = None, None   # YOUR CODE HERE

In [ ]:
kept = draws[1000:]
assert len(draws) == 20_000
assert abs(kept.mean()) < 0.15 and 0.85 < kept.std() < 1.15, (kept.mean(), kept.std())
assert 0.3 < rate < 0.6, f"acceptance rate {rate:.2f}; the theoretical value for this step size is about 0.44"
tiny, tiny_rate = metropolis_1d(lambda z: -0.5 * z ** 2, 0.0, 0.01, 2000, seed=1)
assert tiny_rate > 0.95, "tiny steps are nearly always accepted but explore very slowly"
print(f"✅ Correct! mean {kept.mean():.2f}, sd {kept.std():.2f}, acceptance rate {rate:.2f}")

## Exercise 8

**Convergence diagnostic.** Write `r_hat(chains)` for a 2-D array (rows = chains, columns = draws): $W$ = mean of the within-chain variances (`ddof=1`), $B=L\cdot\operatorname{Var}(\text{chain means})$ (`ddof=1`), and
$\hat R=\sqrt{\big(\tfrac{L-1}{L}W+\tfrac{B}{L}\big)/W}$. Apply it to the well-mixed chains `good` and to the disagreeing chains `bad` (means 0, 1, 2, 3) as `rhat_good`, `rhat_bad`.

In [ ]:
def r_hat(chains):
    # YOUR CODE HERE
    pass

rng = np.random.default_rng(5)
good = rng.normal(0, 1, (4, 2000))
bad = rng.normal(0, 1, (4, 2000)) + np.arange(4)[:, None]
rhat_good = rhat_bad = None   # YOUR CODE HERE

In [ ]:
assert rhat_good < 1.02, rhat_good
assert rhat_bad > 1.5, "chains centred in different places have not converged to the same distribution"
assert r_hat(np.vstack([good[0], good[0]])) < 1.001, "two identical chains cannot disagree"
print(f"✅ Correct! R-hat = {rhat_good:.3f} for the well-mixed chains and {rhat_bad:.2f} for the chains that disagree")

## Exercise 9

**Interpret the interval.** A 95% credible interval for the control conversion rate is (0.109, 0.256), from a flat prior and 17/100 conversions. Which statement is correct? Set `answer` to a letter.

* **a)** Given the data and the prior, there is a 95% probability that the true conversion rate lies between 0.109 and 0.256.
* **b)** 95% of users convert at a rate between 0.109 and 0.256.
* **c)** If we repeated the experiment, 95% of the intervals would contain the true rate, so this interval has no probability meaning.
* **d)** The prior had no effect, so the interval is identical to the frequentist one in every sense.

In [ ]:
answer = None   # "a", "b", "c" or "d"

In [ ]:
assert answer == "a", "A credible interval is a probability statement about the parameter, conditional on the model and the prior"
print("✅ Correct!")